# **Sequências usando RNNs e CNNs**

## **Motivação**

Redes feedforward passam informação em uma única direção: input → camadas → output. Isso funciona bem quando os dados são **independentes** entre si (ex: classificar imagens), mas falha quando existe **dependência sequencial**, prever x sem o contexto de x-1, x-2... é uma desvantagem.

**Exemplo:** em "O moço viu um carro e não atravessou a ___", só sabemos que a resposta é "rua" com o contexto das palavras anteriores e além disso se uma palavra ali troca de posicao tudo muda.

*Vamos usar todo o passado como input entao?*  Sequências têm tamanhos variáveis e passados longos tornam o input inviável.

*resumir o passado com estatísticas (média, moda, mediana...) ?*
Funciona, mas tem limitações:
- **Perde a ordem** dos eventos (ex: [1,2,10] e [10,2,1] têm a mesma média, mas são sequências opostas) 
- Informacao muito antiga pode **enviesar as estatisticas**
- Exige **feature engineering manual** (descobrir na mão o que importa)

→ Troca **capacidade de captar padrões complexos** por **simplicidade/interpretabilidade**.

Ai entra RNNs que criam uma **memória dos inputs anteriores**, armazenada como parâmetros da rede, aprendida automaticamente, sem feature engineering manual. Elas trabalham com sequencias, se antes uma instancia/exemplo era representado por *m* colunas de features, agora uma instancia é representada por *m* vetores (sequencias temporais) de tamanho max_len, ou seja, *m* caracteristicas e suas evoluções e *max_len* timesteps.

## **Neuronios recorrentes e camadas**

A entrada de uma camada recorrente tem sempre 3 dimensões sendo elas [seq_len, batch_size, n_features] por padrão do pytorch e [batch_size, seq_len, n_features] caso o parametro *batch_first = True*. Sendo cada uma dessas dimensões:

- **batch_size:** tamanho do batch da epoca (quantas instancias considerar para cada atualizacao dos parametros em uma epoca)
    - Um exemplo seria 12 clientes computados por vez para computar o gradiente
- **seq_len:** tamanho maximo das sequencias de uma instancia, ou seja, tamanho maximo de passos no tempo
    - Um exemplo seria 12 meses/passos de histórico de cada cliente
    - Em sequencias de tamanho variado, uma vez definido o max_len, todas as sequencias que forem menores que max_len são completadas com padding que são valores artificiais que permitem todas as instancias caberem em um unico tensor comportado. A *função pack_padded_sequence(...)* é uma técnica de otimização que informa à LSTM exatamente onde cada sequência termina, para que ela ignore o padding e não "aprenda" com dados falsos.
- **n_features (input_size):** quantas variaveis existem a cada passo de tempo 
    - Um exemplo seria 3 caracteristicas do cliente em cada mes como gastos, chamadas e usos

Dentro dessa camada existem *hidden_size* neurônios, cada um deles recebe 2 vetores de entrada e tem 2 vetores de pesos:

- ${x_t}$: Vetor de features de uma instancia no tempo t, ou seja, os valores das series temporais no tempo t. Dimensão = ${1 * n_{input}}$
- ${y_{t-1}}$: Vetor de saidas do timestep anterior (uma saida por neuronio). Dimensão = ${1 * n_{neuronios}}$
- ${w_x}$: Vetor de pesos do input. Dimensão = ${1 * n_{input}}$
- ${w_{y_{t-1}}}$:Vetor de pesos do timestep anterior. Dimensão = ${1 * n_{neuronios}}$

A saida de um timestep t para uma instancia de treino é dada pelo 

